In [ ]:
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow import keras
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
from keras.utils import to_categorical

In [ ]:
sample_messages_file_path = r"C:\Users\Lenovo\Desktop\test.txt"
with open(sample_messages_file_path, 'r') as f : samples = f.readlines()

In [ ]:
tokenized_samples = [simple_preprocess(line) for line in samples]

In [ ]:
w2v_model = Word2Vec(sentences=tokenized_samples, vector_size=100, min_count=2, window=5, workers=4)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(samples)
word_to_index = tokenizer.word_index  
vocab_size = len(word_to_index) + 1
embedding_dim = 100

In [ ]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in word_to_index.items():
    if word in w2v_model.wv: embedding_matrix[idx] = w2v_model.wv[word]


In [ ]:
input_sequences = []
for line in samples:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [ ]:
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
X = input_sequences[:, :-1]         
y = input_sequences[:, -1]          
y = to_categorical(y, num_classes=vocab_size)

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True,
        input_length=max_len - 1
    ),
    LSTM(86),
    Dense(vocab_size, activation="softmax")
])

In [ ]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
history = model.fit(X, y, epochs=50, batch_size=32, verbose=1)

In [ ]:

def predict_next_word(seed_text, num_words=1):
    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding="pre")
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]
        output_word = ""
        
        for word, index in word_to_index.items():
            if index == predicted_index:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [ ]:
if __name__ == "__main__" : print(predict_next_word("It's nice", num_words=1))